# Linear static analysis

A `Shell` object describes a plate or a cylindrical shell through its
attributes: the geometry, the laminate, the number of terms `m` and `n` of
the approximation functions and the model, by default
`'plate_clpt_donnell'` for plates and `'cylshell_clpt_sanders'` for
cylindrical shells, see `panels.models` for the other models.
The boundary conditions are controlled by the flags `x1u`, `x1ur`,
`x2u`, ..., `y2wr`, where `0` constrains and `1` releases the
translation (e.g. `x1u`) or the rotation (e.g. `x1ur`) of each
displacement component at each edge, see
[Bardell's functions](../bardell.rst). The models based on shear
deformation theories have the rotations as independent fields, with the
analogous flags `x1phix`, `x1phixr`, ..., `y2phiyr`.

The loads are added with `Shell.add_point_load`,
`Shell.add_distr_load_fixed_x` or `Shell.add_distr_load_fixed_y`,
and the linear system $\boldsymbol{K}_C\boldsymbol{c} = \boldsymbol{F}_{ext}$ is solved with
[structsolve](https://github.com/saullocastro/structsolve). The displacement,
strain and stress fields are calculated from the Ritz constants `c` with
`Shell.uvw`, `Shell.strain` and `Shell.stress`. The code
below is extracted from one of the `panels` unit tests:

In [1]:
import numpy as np
from structsolve import static
from panels.shell import Shell

In [2]:
# from tests/tests_shell/test_field_outputs.py
def test_panel_field_outputs():
    m = 7
    n = 6
    for model in ['plate_clpt_donnell',
                  'cylshell_clpt_donnell',
                  'cylshell_clpt_sanders',
                  'plate_fsdt_donnell',
                  'plate_tsdt_donnell']:
        print('Testing model %s' % model)
        s = Shell()
        s.model = model
        s.x1u = 1
        s.y1u = 1
        s.y2u = 1
        s.x1v = 0
        s.x2v = 0
        s.y1v = 0
        s.y2v = 0

        s.a = 2.
        s.b = 1.
        s.r = 1.e5
        s.stack = [0, -45, +45, 90, +45, -45, 0, 0]
        s.plyt = 1e-3*0.125
        s.laminaprop = (142.5e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 5.1e9)
        s.nx = m
        s.ny = n
        s.m = m
        s.n = n

        P = 1000.
        Nxx = P/s.b
        s.add_distr_load_fixed_x(0, funcx=lambda y: Nxx, funcy=None, funcz=None, cte=False)
        increments, cs = static(s.calc_kC(), s.calc_fext(), silent=True)
        _, fields = s.uvw(cs[0])
        _, fields = s.stress(cs[0])
        _, fields = s.strain(cs[0])
        assert np.isclose(fields.get('w').max(), 0.00014, rtol=0.05)
        assert np.isclose(fields.get('Nxx').min(), -1017.4018, rtol=0.001)
        assert np.isclose(fields.get('eyy').min(), -4.1451e-07, rtol=0.001)

In [3]:
test_panel_field_outputs()
print('test_panel_field_outputs: passed')

Testing model plate_clpt_donnell


Testing model cylshell_clpt_donnell


Testing model cylshell_clpt_sanders


Testing model plate_fsdt_donnell


Testing model plate_tsdt_donnell


test_panel_field_outputs: passed
